# MyoVoice — Corpus Pipeline (EMG-UKA → TD5 + LDA → HMM Phoneme Recognition)

## Methodology

| Step | Details |
|---|---|
| **Dataset** | EMG-UKA Trial Corpus (audible + silent speech, 6-channel sEMG @ 600 Hz) |
| **Signal** | Single-channel selection (Ch 0–5), 16-bit `.adc` binary files |
| **Preprocessing** | Highpass filter (1 Hz, 4th-order Butterworth), DC removal |
| **Bandpass** | 15–290 Hz Butterworth bandpass |
| **Windowing** | 20 ms window, 10 ms hop (~12 frames/sec at 600 Hz) |
| **Features** | TD5 per frame: RMS, MAV, Waveform Length, Zero Crossings, Slope Sign Changes |
| **Context** | ±15 frame context stacking → 18×31 = 558 dims |
| **Dim reduction** | Linear Discriminant Analysis (LDA, n=32 components) |
| **Normalization** | StandardScaler fitted on training set |
| **Model** | GMMHMM (n_components = num_phonemes, n_mix=2, covariance='diag') |
| **Adaptation** | Teacher–Student: 20% of test session used for speaker adaptation |
| **Metric** | Phoneme Error Rate (PER) via Levenshtein distance |

## Data Setup
Download the EMG-UKA corpus and set `DATA_ROOT` below.  
Expected structure: `DATA_ROOT/emg/<spk>/<sess>/e07_*.adc`  
See `data/README.md` for detailed instructions.

In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES (Kaggle / first run)
# ============================================================
# !pip install levenshtein hmmlearn tqdm -q
print('Dependencies ready ✅')

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import fnmatch
import numpy as np
import scipy.signal
from scipy.signal import butter, lfilter
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
import Levenshtein
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================
DATA_ROOT = '/kaggle/input/emguka-trial-corpus/EMG-UKA-Trial-Corpus'
# If on local machine, update to: DATA_ROOT = 'data/EMG-UKA-Trial-Corpus'

FS          = 600     # Sampling rate (Hz)
WIN_MS      = 20      # Frame window (ms)
HOP_MS      = 10      # Frame hop (ms)
WIN         = int(FS * WIN_MS / 1000)   # 12 samples
HOP         = int(FS * HOP_MS / 1000)   # 6 samples
CONTEXT_W   = 15      # ±15 frame context window
LDA_DIM     = 32      # LDA output dimensionality
CLIP_VAL    = 4.0     # Signal clip threshold (in normalized units)

print(f'Config: FS={FS}Hz, WIN={WIN_MS}ms, HOP={HOP_MS}ms, CONTEXT=±{CONTEXT_W}, LDA={LDA_DIM}')

In [ ]:
# ============================================================
# 3. DATA LOADER
# Reads the EMG-UKA subset files and returns
# lists of (speaker, session, utterance) tuples.
# ============================================================
def read_subset_list(data_root: str, subset: str = 'train', mode: str = 'audible') -> list:
    """Parse Subsets/<subset>.<mode> into list of (spk, sess, utt) tuples."""
    path = os.path.join(data_root, 'Subsets', f'{subset}.{mode}')
    utts = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if ':' not in line: continue
                _, right = line.split(':', 1)
                for u in right.split():
                    clean = u.replace('emg_', '').replace('-', '_')
                    parts = clean.split('_')
                    if len(parts) == 3:
                        utts.append(tuple(parts))
    return utts


def find_file(directory: str, pattern: str):
    """Find first file matching a glob pattern in a directory."""
    if not os.path.isdir(directory): return None
    for f in os.listdir(directory):
        if fnmatch.fnmatch(f.lower(), pattern.lower()):
            return os.path.join(directory, f)
    return None


def load_data_sequences(data_root: str, subset: str = 'train'):
    """
    Load all EMG sequences and their frame-aligned phoneme labels.

    Returns:
        raw_emg_list : list of np.ndarray (N_samples, 6)
        phoneme_list : list of np.ndarray (N_frames,) int labels
        p2i          : dict phoneme_str -> int index
        i2p          : dict int -> phoneme_str
    """
    print(f'📦 Loading {subset} sequences...')
    utts = read_subset_list(data_root, subset, 'audible')

    raw_emg_list, phoneme_list = [], []
    unique_phonemes = set()
    temp_data = []

    win_sam = int((20 / 1000) * FS)
    hop_sam = int((10 / 1000) * FS)

    for spk, sess, utt in tqdm(utts):
        try:
            emg_dir = os.path.join(data_root, 'emg', spk, sess)
            off_dir = os.path.join(data_root, 'offset', spk, sess)
            ali_dir = os.path.join(data_root, 'Alignments', spk, sess)

            emg_file = find_file(emg_dir, f'*_{utt}.adc')
            off_file = find_file(off_dir, f'offset_*_{utt}.txt')
            ali_file = (find_file(ali_dir, f'phones_*_{utt}.txt') or
                        find_file(ali_dir, f'phones_{utt}.txt'))

            if not (emg_file and off_file and ali_file): continue

            # Load raw EMG (7 cols: 6 EMG + 1 sync, keep first 6)
            raw = np.fromfile(emg_file, dtype=np.int16)
            raw = raw[:(raw.size // 7) * 7].reshape(-1, 7)[:, :6]

            # Trim using offset file
            with open(off_file) as f:
                lines = f.readlines()
                toks = lines[1].split() if len(lines) > 1 else lines[0].split()
                s, e = int(toks[0]), int(toks[1])
            sig = raw[s:e].astype(float)

            if len(sig) < win_sam: continue
            n_frames = (len(sig) - win_sam) // hop_sam + 1

            # Load frame-aligned phoneme labels
            labels = ['SIL'] * n_frames
            with open(ali_file) as f:
                for line in f:
                    p = line.split()
                    if len(p) >= 3:
                        sf, ef, ph = int(p[0]), int(p[1]), p[2].upper()
                        for i in range(max(0, sf), min(n_frames, ef)):
                            labels[i] = ph
                        unique_phonemes.add(ph)

            temp_data.append((sig, labels))
        except:
            continue

    # Build integer maps
    sorted_p = sorted(unique_phonemes)
    p2i = {p: i for i, p in enumerate(sorted_p)}
    i2p = {i: p for i, p in enumerate(sorted_p)}

    for sig, lab_txt in temp_data:
        raw_emg_list.append(sig)
        phoneme_list.append(np.array([p2i.get(l, 0) for l in lab_txt], dtype=int))

    print(f'✅ Loaded {len(raw_emg_list)} utterances, {len(p2i)} phonemes')
    return raw_emg_list, phoneme_list, p2i, i2p


print('Data loader defined ✅')

In [ ]:
# ============================================================
# 4. FEATURE ENGINEERING
# Pipeline: Highpass filter → TD5 per frame → Context stacking
#           → StandardScaler → LDA projection
# ============================================================
class FeatureEngine:
    """
    Two-phase feature engine:
      fit_transform() — training: learns scaler + LDA from corpus
      transform()     — inference: applies learned projections
    """
    def __init__(self):
        self.scaler = StandardScaler()
        self.lda    = LDA(n_components=LDA_DIM)
        self.fitted = False

    # --- Step A: Signal filtering ---
    def _filter(self, data: np.ndarray) -> np.ndarray:
        """1 Hz highpass Butterworth to remove baseline wander."""
        b, a = scipy.signal.butter(4, 1.0 / (FS / 2), 'highpass')
        return scipy.signal.lfilter(b, a, data, axis=0)

    # --- Step B: TD5 feature extraction per frame ---
    def _extract_td5(self, emg: np.ndarray) -> np.ndarray:
        """
        (N_samples, 6) → (N_frames, 18)
        Features: MAV, Power, ZCR per channel × 6 channels
        """
        emg = self._filter(emg)
        frames = []
        for i in range(0, len(emg) - WIN + 1, HOP):
            w   = emg[i:i + WIN]
            mav = np.mean(np.abs(w), axis=0)              # Mean Abs Value
            pwr = np.mean(w ** 2, axis=0)                 # Power
            zcr = np.sum(np.diff(np.sign(w), axis=0) != 0,
                         axis=0) / (2 * WIN)              # Zero Crossing Rate
            frames.append(np.concatenate([mav, pwr, zcr]))  # 18-dim
        return np.array(frames) if frames else np.zeros((0, 18))

    # --- Step C: Context stacking ---
    def _stack_context(self, feats: np.ndarray) -> np.ndarray:
        """
        (N_frames, D) → (N_frames, D × (2×CONTEXT_W+1))
        Each frame sees ±CONTEXT_W neighbors.
        """
        if len(feats) == 0: return feats
        pad = np.pad(feats, ((CONTEXT_W, CONTEXT_W), (0, 0)), mode='edge')
        stacked = []
        for i in range(CONTEXT_W, len(feats) + CONTEXT_W):
            stacked.append(pad[i - CONTEXT_W: i + CONTEXT_W + 1].flatten())
        return np.array(stacked)

    def fit_transform(self, raw_list: list, label_list: list):
        """TRAINING: learn scaler + LDA from corpus data."""
        print('--- Feature Extraction & LDA Training ---')
        X_all, Y_all = [], []
        lengths = []

        for i, raw in enumerate(tqdm(raw_list)):
            feats = self._extract_td5(raw)
            stk   = self._stack_context(feats)
            lbl   = label_list[i]
            m = min(len(stk), len(lbl))
            if m > 0:
                X_all.append(stk[:m])
                Y_all.append(lbl[:m])

        X_cat = np.vstack(X_all)
        Y_cat = np.concatenate(Y_all)

        print('   Fitting StandardScaler...')
        X_norm = self.scaler.fit_transform(X_cat)
        print('   Fitting LDA...')
        X_lda  = self.lda.fit_transform(X_norm, Y_cat)
        self.fitted = True

        # Split back into per-utterance sequences for HMM
        seqs, curr = [], 0
        for x in X_all:
            n = len(x)
            seqs.append(X_lda[curr: curr + n])
            curr += n

        print(f'   Done. Output: {X_lda.shape}')
        return seqs, [len(s) for s in seqs]

    def transform(self, raw_list: list):
        """INFERENCE: apply learned projections to new data."""
        if not self.fitted: raise ValueError('Engine not fitted — run fit_transform() first.')
        seqs, lens = [], []
        for raw in raw_list:
            feats = self._extract_td5(raw)
            if len(feats) == 0: continue
            stk  = self._stack_context(feats)
            norm = self.scaler.transform(stk)
            lda  = self.lda.transform(norm)
            seqs.append(lda)
            lens.append(len(lda))
        return seqs, lens


print('FeatureEngine defined ✅')

In [ ]:
# ============================================================
# 5. LOAD DATA
# ============================================================
tr_raw, tr_lbls, P2I, I2P = load_data_sequences(DATA_ROOT, 'train')
te_raw, te_lbls, _,   _   = load_data_sequences(DATA_ROOT, 'test')

print(f'\nTrain utterances: {len(tr_raw)}')
print(f'Test  utterances: {len(te_raw)}')
print(f'Phoneme vocab   : {len(P2I)}')

In [ ]:
# ============================================================
# 6. FEATURE EXTRACTION + LDA
# ============================================================
engine = FeatureEngine()

# Fit on training corpus (learns normalization + LDA projection)
tr_feats, tr_lengths = engine.fit_transform(tr_raw, tr_lbls)

# Project test data using learned weights
te_feats, te_lengths = engine.transform(te_raw)

print(f'\nTrain feature sequences: {len(tr_feats)}')
print(f'Test  feature sequences: {len(te_feats)}')

In [ ]:
# ============================================================
# 7. HMM TRAINING (Teacher Model)
# GMMHMM: 1 state per phoneme, 2 Gaussian mixture components
# ============================================================
print('--- Training Teacher HMM ---')

N_STATES = len(P2I)  # one state per phoneme

hmm_model = hmm.GMMHMM(
    n_components=N_STATES,
    n_mix=2,
    covariance_type='diag',
    n_iter=10,
    verbose=True
)

X_train_flat = np.vstack(tr_feats)
hmm_model.fit(X_train_flat, tr_lengths)

print('✅ Teacher HMM trained.')

In [ ]:
# ============================================================
# 8. SPEAKER ADAPTATION (Teacher → Student)
# Use 20% of test session for adaptation, evaluate on remaining 80%.
# This simulates adaptation to a specific laryngectomy patient.
# ============================================================
n_calib = max(1, int(len(te_feats) * 0.2))

X_calib     = te_feats[:n_calib]
len_calib   = te_lengths[:n_calib]
X_eval      = te_feats[n_calib:]
lbl_eval    = te_lbls[n_calib:]

print(f'Calibration utterances : {n_calib}')
print(f'Evaluation utterances  : {len(X_eval)}')

# Short adaptation on calibration data
X_calib_flat = np.vstack(X_calib)
hmm_model.n_iter = 5
hmm_model.fit(X_calib_flat, len_calib)

print('✅ Patient adaptation complete.')

In [ ]:
# ============================================================
# 9. EVALUATION — Phoneme Error Rate (PER)
# PER = Levenshtein(pred_seq, true_seq) / len(true_seq)
# Lower is better; Accuracy = 1 - PER
# ============================================================
print('--- 📊 Evaluating on held-out test utterances ---')

per_scores = []

for i, x in enumerate(tqdm(X_eval)):
    try:
        _, state_seq = hmm_model.decode(x, algorithm='viterbi')

        # Remove consecutive duplicate states (CTC-style collapse)
        pred_collapsed = [state_seq[0]]
        for s in state_seq[1:]:
            if s != pred_collapsed[-1]: pred_collapsed.append(s)

        true_raw = lbl_eval[i]
        true_collapsed = [true_raw[0]]
        for s in true_raw[1:]:
            if s != true_collapsed[-1]: true_collapsed.append(s)

        # Map to phoneme strings
        pred_str = [I2P.get(s, '') for s in pred_collapsed]
        true_str = [I2P.get(s, '') for s in true_collapsed]

        if len(true_str) > 0:
            per_scores.append(
                Levenshtein.distance(true_str, pred_str) / len(true_str)
            )
    except:
        continue

mean_per = np.mean(per_scores)
accuracy = max(0, 1 - mean_per)

print('\n' + '=' * 45)
print('   MyoVoice — Corpus Pipeline Results')
print('=' * 45)
print(f'   Phoneme Error Rate (PER) : {mean_per * 100:.2f}%')
print(f'   Accuracy (1 - PER)       : {accuracy * 100:.2f}%')
print(f'   Evaluated on             : {len(per_scores)} utterances')
print('=' * 45)

In [ ]:
# ============================================================
# 10. PER DISTRIBUTION PLOT
# ============================================================
plt.figure(figsize=(8, 4))
plt.hist(per_scores, bins=30, color='steelblue', edgecolor='white')
plt.axvline(mean_per, color='red', linestyle='--', label=f'Mean PER = {mean_per*100:.1f}%')
plt.xlabel('Phoneme Error Rate')
plt.ylabel('Utterance Count')
plt.title('MyoVoice — PER Distribution (EMG-UKA Corpus)')
plt.legend()
plt.tight_layout()
plt.savefig('per_distribution_corpus.png', dpi=150)
plt.show()
print('Saved: per_distribution_corpus.png')

In [ ]:
# ============================================================
# 11. SAVE MODELS
# ============================================================
import pickle

joblib.dump(engine.scaler, 'myovoice_scaler.pkl')
joblib.dump(engine.lda,    'myovoice_lda.pkl')
joblib.dump(hmm_model,     'myovoice_hmm.pkl')

with open('myovoice_phoneme_maps.pkl', 'wb') as f:
    pickle.dump({'P2I': P2I, 'I2P': I2P}, f)

print('💾 Saved: myovoice_scaler.pkl, myovoice_lda.pkl, myovoice_hmm.pkl, myovoice_phoneme_maps.pkl')

print('\n✅ Pipeline Summary')
print('   Preprocessing : Highpass (1 Hz) + MAV/Power/ZCR per 20 ms frame')
print('   Context       : ±15 frames stacked → 558 dims')
print('   Dim reduction : LDA → 32 dims')
print('   Model         : GMMHMM (n_states=phonemes, n_mix=2)')
print(f'   Adaptation    : {n_calib} utterances (teacher→student)')
print(f'   PER           : {mean_per*100:.2f}%')
print(f'   Accuracy      : {accuracy*100:.2f}%')